# Student Health Risk Prediction

This notebook performs EDA, feature engineering, validation, model training, and submission-file generation for `train.csv`, `test.csv`, and `sample_submission.csv`.

Generated files are written under `submissions/`, with one separate submission CSV for each model.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

RANDOM_STATE = 42
TARGET = 'health_condition'
ID_COL = 'id'

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'submissions'
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = ROOT / 'train.csv'
TEST_PATH = ROOT / 'test.csv'
SAMPLE_SUBMISSION_PATH = ROOT / 'sample_submission.csv'

print('Working directory:', ROOT)
print('Outputs directory:', OUTPUT_DIR)

## 1. Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Sample submission shape:', sample_submission.shape)
print('Sample submission columns:', sample_submission.columns.tolist())

assert TARGET in train_df.columns, f'Missing target column: {TARGET}'
assert list(sample_submission.columns) == [ID_COL, TARGET], 'Unexpected sample submission format'
assert test_df[ID_COL].equals(sample_submission[ID_COL]), 'test.csv ids do not match sample_submission.csv ids'

display(train_df.head())
display(test_df.head())
display(sample_submission.head())

## 2. EDA

In [ ]:
target_distribution = (
    train_df[TARGET]
    .value_counts()
    .to_frame('count')
    .assign(percent=lambda x: (x['count'] / len(train_df) * 100).round(2))
)

display(target_distribution)

ax = target_distribution['count'].plot(kind='bar', figsize=(7, 4), title='Target Distribution')
ax.set_xlabel(TARGET)
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
missing_train = (
    train_df.isna().sum()
    .to_frame('missing_count')
    .assign(missing_percent=lambda x: (x['missing_count'] / len(train_df) * 100).round(2))
    .sort_values('missing_count', ascending=False)
)
missing_test = (
    test_df.isna().sum()
    .to_frame('missing_count')
    .assign(missing_percent=lambda x: (x['missing_count'] / len(test_df) * 100).round(2))
    .sort_values('missing_count', ascending=False)
)

print('Missing values in train:')
display(missing_train)
print('Missing values in test:')
display(missing_test)

In [ ]:
feature_cols = [c for c in train_df.columns if c not in [ID_COL, TARGET]]
numeric_cols_raw = train_df[feature_cols].select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_cols_raw = [c for c in feature_cols if c not in numeric_cols_raw]

print('Numeric columns:', numeric_cols_raw)
print('Categorical columns:', categorical_cols_raw)

display(train_df[numeric_cols_raw].describe().T.round(3))

display(
    pd.DataFrame({
        'column': categorical_cols_raw,
        'unique_values_including_missing': [train_df[c].nunique(dropna=False) for c in categorical_cols_raw],
        'top_value': [train_df[c].mode(dropna=True).iloc[0] if not train_df[c].mode(dropna=True).empty else np.nan for c in categorical_cols_raw],
    })
)

In [ ]:
# Class-wise numeric means help reveal coarse signal before modeling.
class_numeric_means = train_df.groupby(TARGET)[numeric_cols_raw].mean(numeric_only=True).round(3)
display(class_numeric_means)

# Simple histograms for the core numeric features.
train_df[numeric_cols_raw].hist(figsize=(12, 8), bins=30)
plt.suptitle('Numeric Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

The engineered features combine raw health/activity signals, missingness, and binned categories. Missing values are still handled inside each model pipeline to avoid leakage.

In [ ]:
def safe_divide(numerator, denominator, fill_value=np.nan):
    denominator = denominator.replace(0, np.nan) if isinstance(denominator, pd.Series) else denominator
    result = numerator / denominator
    return result.replace([np.inf, -np.inf], np.nan).fillna(fill_value)


def add_features(df):
    out = df.copy()
    numeric_base = [
        'sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
        'step_count', 'exercise_duration', 'water_intake'
    ]
    available_numeric = [col for col in numeric_base if col in out.columns]
    out['missing_feature_count'] = out[available_numeric].isna().sum(axis=1)

    out['steps_per_exercise_min'] = safe_divide(out['step_count'], out['exercise_duration'] + 1)
    out['calories_per_step'] = safe_divide(out['calorie_expenditure'], out['step_count'] + 1)
    out['calories_per_exercise_min'] = safe_divide(out['calorie_expenditure'], out['exercise_duration'] + 1)
    out['water_per_bmi'] = safe_divide(out['water_intake'], out['bmi'])
    out['sleep_bmi_interaction'] = out['sleep_duration'] * out['bmi']
    out['heart_rate_bmi_interaction'] = out['heart_rate'] * out['bmi']
    out['sleep_deficit_from_8h'] = (8 - out['sleep_duration']).clip(lower=0)
    out['sleep_excess_over_9h'] = (out['sleep_duration'] - 9).clip(lower=0)
    out['activity_score'] = (
        safe_divide(out['step_count'], pd.Series(10000, index=out.index))
        + safe_divide(out['exercise_duration'], pd.Series(60, index=out.index))
        + safe_divide(out['calorie_expenditure'], pd.Series(2500, index=out.index))
        + safe_divide(out['water_intake'], pd.Series(2, index=out.index))
    )

    out['bmi_category'] = pd.cut(
        out['bmi'],
        bins=[-np.inf, 18.5, 25, 30, np.inf],
        labels=['underweight', 'normal', 'overweight', 'obese'],
    ).astype('object')
    out['sleep_duration_bin'] = pd.cut(
        out['sleep_duration'],
        bins=[-np.inf, 5, 7, 9, np.inf],
        labels=['very_short', 'short', 'recommended', 'long'],
    ).astype('object')
    out['step_count_bin'] = pd.cut(
        out['step_count'],
        bins=[-np.inf, 5000, 10000, 15000, np.inf],
        labels=['low', 'moderate', 'high', 'very_high'],
    ).astype('object')
    out['water_intake_bin'] = pd.cut(
        out['water_intake'],
        bins=[-np.inf, 1.5, 2.5, 3.5, np.inf],
        labels=['low', 'moderate', 'high', 'very_high'],
    ).astype('object')
    out['heart_rate_bin'] = pd.cut(
        out['heart_rate'],
        bins=[-np.inf, 60, 80, 100, np.inf],
        labels=['low', 'normal', 'elevated', 'high'],
    ).astype('object')

    return out


train_fe = add_features(train_df)
test_fe = add_features(test_df)

print('Train shape after feature engineering:', train_fe.shape)
print('Test shape after feature engineering:', test_fe.shape)
display(train_fe.head())

In [ ]:
def split_columns(df):
    feature_columns = [col for col in df.columns if col not in {ID_COL, TARGET}]
    numeric_columns = df[feature_columns].select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_columns = [col for col in feature_columns if col not in numeric_columns]
    return numeric_columns, categorical_columns


numeric_cols, categorical_cols = split_columns(train_fe)
print(f'Numeric model features: {len(numeric_cols)}')
print(f'Categorical model features: {len(categorical_cols)}')
print('Categorical features:', categorical_cols)

In [ ]:
eda_report_path = OUTPUT_DIR / 'eda_summary.md'

eda_lines = [
    '# Student Health Risk EDA Summary',
    '',
    f'- Train shape: `{train_df.shape[0]:,}` rows x `{train_df.shape[1]:,}` columns',
    f'- Test shape: `{test_df.shape[0]:,}` rows x `{test_df.shape[1]:,}` columns',
    f'- Target column: `{TARGET}`',
    '',
    '## Target Distribution',
    target_distribution.to_markdown(),
    '',
    '## Missing Values - Train',
    missing_train.to_markdown(),
    '',
    '## Missing Values - Test',
    missing_test.to_markdown(),
    '',
    '## Numeric Feature Summary After Feature Engineering',
    train_fe[numeric_cols].describe().T.round(3).to_markdown(),
    '',
    '## Categorical Feature Cardinality After Feature Engineering',
    pd.Series({col: train_fe[col].nunique(dropna=False) for col in categorical_cols})
        .sort_values(ascending=False)
        .to_frame('unique_values_including_missing')
        .to_markdown(),
    '',
]

eda_report_path.write_text('\n'.join(eda_lines), encoding='utf-8')
print('Wrote:', eda_report_path)

## 4. Preprocessing and Model Definitions

The notebook trains several different model families:

- Logistic Regression
- Balanced Logistic Regression
- Hist Gradient Boosting
- Random Forest
- Extra Trees

Each model gets its own labeled submission file. Kaggle public scores favored the class-weighted tree submissions, so the default random forest keeps `class_weight='balanced_subsample'`.

In [ ]:
def make_onehot_preprocessor(numeric_columns, categorical_columns, scale_numeric):
    numeric_steps = [('imputer', SimpleImputer(strategy='median', add_indicator=True))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))

    categorical_pipeline = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ('num', Pipeline(numeric_steps), numeric_columns),
            ('cat', categorical_pipeline, categorical_columns),
        ]
    )


def make_ordinal_preprocessor(numeric_columns, categorical_columns):
    return ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median', add_indicator=True), numeric_columns),
            (
                'cat',
                Pipeline(
                    steps=[
                        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
                    ]
                ),
                categorical_columns,
            ),
        ]
    )


linear_preprocessor = make_onehot_preprocessor(numeric_cols, categorical_cols, scale_numeric=True)
tree_preprocessor = make_onehot_preprocessor(numeric_cols, categorical_cols, scale_numeric=False)
ordinal_preprocessor = make_ordinal_preprocessor(numeric_cols, categorical_cols)

models = {
    'logistic_regression': Pipeline(
        steps=[
            ('preprocess', linear_preprocessor),
            ('model', LogisticRegression(max_iter=300, n_jobs=-1, random_state=RANDOM_STATE)),
        ]
    ),
    'balanced_logistic_regression': Pipeline(
        steps=[
            ('preprocess', linear_preprocessor),
            ('model', LogisticRegression(max_iter=300, n_jobs=-1, random_state=RANDOM_STATE, class_weight='balanced')),
        ]
    ),
    'hist_gradient_boosting': Pipeline(
        steps=[
            ('preprocess', ordinal_preprocessor),
            ('model', HistGradientBoostingClassifier(
                learning_rate=0.06,
                max_iter=220,
                max_leaf_nodes=31,
                l2_regularization=0.05,
                early_stopping=True,
                random_state=RANDOM_STATE,
            )),
        ]
    ),
    'random_forest': Pipeline(
        steps=[
            ('preprocess', tree_preprocessor),
            ('model', RandomForestClassifier(
                n_estimators=120,
                max_depth=18,
                min_samples_leaf=10,
                class_weight='balanced_subsample',
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )),
        ]
    ),
    'extra_trees': Pipeline(
        steps=[
            ('preprocess', tree_preprocessor),
            ('model', ExtraTreesClassifier(
                n_estimators=160,
                max_depth=22,
                min_samples_leaf=8,
                class_weight='balanced',
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )),
        ]
    ),
}

list(models.keys())

## 5. Train, Validate, and Generate Submission Files

The validation split is stratified so each class keeps approximately the same percentage as the full training data. After validation, each model is refit on the full training dataset before predicting `test.csv`.

In [ ]:
X = train_fe.drop(columns=[TARGET])
y = train_fe[TARGET]
X_test = test_fe.copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.18,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Training rows:', X_train.shape[0])
print('Validation rows:', X_valid.shape[0])
print('Test rows:', X_test.shape[0])

In [ ]:
score_rows = []
submission_paths = []

for model_name, pipeline in models.items():
    print(f'\n=== {model_name} ===')

    validation_model = clone(pipeline)
    validation_model.fit(X_train, y_train)
    valid_pred = validation_model.predict(X_valid)

    report = classification_report(y_valid, valid_pred, output_dict=True, zero_division=0)
    accuracy = accuracy_score(y_valid, valid_pred)
    macro_f1 = f1_score(y_valid, valid_pred, average='macro')
    weighted_f1 = f1_score(y_valid, valid_pred, average='weighted')

    score_rows.append({
        'model': model_name,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'fit_support': len(X_train),
        'validation_support': len(X_valid),
        'per_class_report_json': json.dumps(report, sort_keys=True),
    })

    print(f'Validation accuracy: {accuracy:.5f}')
    print(f'Validation macro F1: {macro_f1:.5f}')
    print(f'Validation weighted F1: {weighted_f1:.5f}')

    final_model = clone(pipeline)
    final_model.fit(X, y)
    test_pred = final_model.predict(X_test)

    submission = sample_submission.copy()
    submission[TARGET] = test_pred
    submission_path = OUTPUT_DIR / f'submission_{model_name}.csv'
    submission.to_csv(submission_path, index=False)
    submission_paths.append(submission_path)
    print('Wrote submission:', submission_path.name)

scores_df = pd.DataFrame(score_rows).sort_values(['accuracy', 'macro_f1'], ascending=False)
scores_path = OUTPUT_DIR / 'validation_scores.csv'
scores_df.to_csv(scores_path, index=False)

print('\nWrote scores:', scores_path)
display(scores_df[['model', 'accuracy', 'macro_f1', 'weighted_f1', 'fit_support', 'validation_support']])

## 6. Check Generated Submission Files

In [ ]:
for path in submission_paths:
    sub = pd.read_csv(path)
    print(path.name, sub.shape, sub[TARGET].value_counts().to_dict())
    assert list(sub.columns) == [ID_COL, TARGET]
    assert len(sub) == len(sample_submission)
    assert sub[ID_COL].equals(sample_submission[ID_COL])

print('\nSubmission files are ready in:', OUTPUT_DIR)
print('Validation scores saved to:', scores_path)
print('EDA report saved to:', eda_report_path)